# 🔮 FORESIGHT — 08: Risk Engine, Prescriptive Actions & What-If Simulations

This notebook demonstrates:
- **Unit Loss Function & Expected Lost Sales Modeling**
- **Financial Exposure Quantification** (Lost Margin vs Excess Holding Penalty)
- **Prescriptive Replenishment Orders & Lateral Transfers**
- **Interactive What-If Disruption Scenario Simulations**

In [ ]:
import pandas as pd
import numpy as np
from foresight.inventory.optimizer import InventoryOptimizer
from foresight.inventory.schema import InventoryParameters
from foresight.risk.scorer import assess_sku_risk, standard_normal_loss_function
from foresight.risk.prescriptive import PrescriptiveEngine
from foresight.risk.simulator import WhatIfSimulator
from foresight.risk.schema import ScenarioParameters

# 1. Unit Loss Function L(z) Demonstration
z_values = [-2.0, -1.0, 0.0, 1.0, 1.645, 2.0, 3.0]
print("=== STANDARD NORMAL UNIT LOSS FUNCTION L(z) ===")
for z in z_values:
    print(f"z = {z:>6.3f} -> L(z) = {standard_normal_loss_function(z):.4f}")

## 2. SKU Risk Quantification and Prescriptive Action Synthesis

In [ ]:
params = InventoryParameters(
    sku_id="SKU-1006",
    store_id="STORE-001",
    current_on_hand=115.0,
    units_on_order=0.0,
    backorders=0.0,
    unit_cost=35.0,
    unit_price=75.0,
    lead_time_days=7.0,
    lead_time_std_days=1.0,
    holding_cost_annual_rate=0.20,
    fixed_order_cost=50.0,
    min_order_qty=20.0,
    target_service_level=0.95,
    forecast_daily_demand_mean=25.0,
    forecast_daily_demand_std=5.0,
)

optimizer = InventoryOptimizer()
opt_res = optimizer.optimize_sku(params)
risk_res = assess_sku_risk(opt_res, params)

engine = PrescriptiveEngine()
rec = engine.generate_recommendation(opt_res, risk_res, params)

print("\n=== PRESCRIPTIVE RECOMMENDATION ===")
print(rec.model_dump_json(indent=2))

## 3. What-If Disruption Scenario Simulator
Simulate a +50% supplier lead time disruption alongside a +20% demand surge.

In [ ]:
scenario = ScenarioParameters(
    scenario_name="Red Sea Shipping Bottleneck + Q4 Demand Surge",
    lead_time_multiplier=1.50,
    demand_multiplier=1.20,
    target_service_level=0.98,
)

simulator = WhatIfSimulator()
sim_result = simulator.simulate_sku(params, scenario)

print("\n=== SCENARIO STRESS TEST RESULT ===")
print(f"Baseline Safety Stock:  {sim_result.baseline_safety_stock:.1f} -> Simulated: {sim_result.simulated_safety_stock:.1f} (Delta: {sim_result.delta_safety_stock:+.1f})")
print(f"Baseline Reorder Point: {sim_result.baseline_reorder_point:.1f} -> Simulated: {sim_result.simulated_reorder_point:.1f} (Delta: {sim_result.delta_reorder_point:+.1f})")
print(f"Baseline Working Cap:  ${sim_result.baseline_working_capital:,.2f} -> Simulated: ${sim_result.simulated_working_capital:,.2f} (Delta: ${sim_result.delta_working_capital:+,.2f})")
print(f"Baseline Stockout Risk: {sim_result.baseline_stockout_risk*100:.1f}% -> Simulated: {sim_result.simulated_stockout_risk*100:.1f}% (Delta: {sim_result.delta_stockout_risk*100:+.1f}%)")